# LeetCode 1016 — Binary String With Substrings Representing 1 To N bằng Python

Link đề: https://leetcode.com/problems/binary-string-with-substrings-representing-1-to-n/ · Độ khó: **Medium** · Chủ đề: String (chuỗi), Bit Manipulation (thao tác bit), Hash Table (bảng băm), Sliding Window (cửa sổ trượt)

Skeleton LeetCode:

```python
class Solution:
    def queryString(self, s: str, n: int) -> bool:
        ...
```

Ghi chú: cho chuỗi nhị phân `s` và số nguyên dương `n`. Hỏi: biểu diễn nhị phân của **mọi** số nguyên trong `[1, n]` có là **chuỗi con liên tiếp** của `s` không? Điểm mấu chốt (một "brain teaser"): `s` chỉ dài tối đa `1000` nên chỉ có tối đa `1000` chuỗi con cho mỗi độ dài, trong khi `n` có thể tới `10^9`. Do đó khi `n` lớn đáp án chắc chắn là `false` (nguyên lý chuồng bồ câu), và bài toán thật sự chỉ cần kiểm tra các `n` nhỏ. Hơn nữa, nếu số `k` có mặt thì số `k // 2` (bỏ bit cuối) cũng có mặt, nên chỉ cần kiểm tra **nửa trên** `(n // 2, n]`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: kiểm tra mọi biểu diễn nhị phân `bin(k)` với `k` từ `1` đến `n` có xuất hiện trong `s` dưới dạng chuỗi con.
2. Nắm các công cụ Python cần thiết: `bin(k)[2:]` (bỏ tiền tố `0b`), toán tử `in` cho chuỗi con, `int(chuỗi, 2)`, `int.bit_length()`, `set`, và dịch bit `k >> 1`.
3. Hiểu hai ý chính: (a) **chặn** `n` bằng nguyên lý chuồng bồ câu (đủ nhiều số nhưng quá ít chuỗi con); (b) **mệnh đề tiền tố**: `bin(k >> 1)` là tiền tố của `bin(k)`, nên chỉ cần kiểm tra nửa trên của đoạn.
4. Đi qua 4 cách: thử từng số bằng `in` (brute force, chỉ cho `n` nhỏ), thử từng số kèm cắt nhanh bằng chuồng bồ câu, thu thập mọi giá trị chuỗi con của `s` vào một `set` rồi so với `n` (cách nộp chính), và chỉ kiểm tra nửa trên `(n // 2, n]`.
5. Tránh các lỗi kinh điển: quên bỏ tiền tố `0b`, chỉ kiểm tra `bin(n)`, lệch biên `range(1, n)`, đếm cả số `0` từ chuỗi con toàn `0`, và chỉ kiểm tra nửa **dưới** thay vì nửa trên.

## 2. Đề bài & ví dụ

Cho chuỗi nhị phân `s` và số nguyên dương `n`, trả về `true` nếu **biểu diễn nhị phân của mọi số nguyên trong đoạn `[1, n]` đều là chuỗi con** của `s`, ngược lại trả về `false`.

Chuỗi con là dãy ký tự **liên tiếp** trong một chuỗi.

**Ví dụ 1:** `s = "0110"`, `n = 3` -> `true`. Các biểu diễn: `1 -> "1"`, `2 -> "10"`, `3 -> "11"`: cả ba đều có trong `"0110"`.

**Ví dụ 2:** `s = "0110"`, `n = 4` -> `false`. `4 -> "100"` không có trong `"0110"`.

**Ràng buộc:**

- `1 <= s.length <= 1000`
- `s[i]` là `'0'` hoặc `'1'`.
- `1 <= n <= 10^9`

In [1]:
from typing import List, Set
import random, time

examples = [
    ("0110", 3, True),
    ("0110", 4, False),
]
for s, n, expected in examples:
    print(f"s={s!r} n={n} -> kỳ vọng {expected}")

s='0110' n=3 -> kỳ vọng True
s='0110' n=4 -> kỳ vọng False


## 3. Kiến thức nền

### 3.1. Biểu diễn nhị phân trong Python

`bin(5)` cho `'0b101'`; phần `[2:]` bỏ tiền tố `0b` được `'101'`. Biểu diễn nhị phân **không có số `0` đứng đầu** (trừ chính số `0`). Ngược lại `int("101", 2) == 5`, và `int("0101", 2) == 5` (số `0` đứng đầu bị bỏ qua).

In [2]:
print(bin(5), bin(5)[2:], int("101", 2), int("0101", 2))
print([bin(k)[2:] for k in range(1, 9)])
print("'0b101' in '0b101' ->", "0b101" in "1010", "| '101' in '1010' ->", "101" in "1010")

0b101 101 5 5
['1', '10', '11', '100', '101', '110', '111', '1000']
'0b101' in '0b101' -> False | '101' in '1010' -> True


### 3.2. Mệnh đề tiền tố: `k // 2` nằm trong `k`

`bin(k >> 1)` chính là `bin(k)` **bỏ bit cuối** (với `k >= 2`). Vậy nếu `bin(k)` là chuỗi con của `s` thì `bin(k >> 1)` cũng là chuỗi con (nó là tiền tố của một chuỗi con). Suy ra: nếu mọi số trong `(n // 2, n]` có mặt thì **mọi** số trong `[1, n]` có mặt, vì mỗi `j <= n // 2` được suy ra từ `k = j * 2^t` thuộc `(n // 2, n]` bằng cách bỏ `t` bit cuối.

In [3]:
for k in (2, 5, 6, 13):
    print(f"bin({k}) = {bin(k)[2:]:6} -> bin({k} >> 1) = bin({k >> 1}) = {bin(k >> 1)[2:]}")
n = 13
cover = {j for j in range(1, n + 1) if j <= n // 2}
reached = set()
for k in range(n // 2 + 1, n + 1):
    j = k
    while j >= 1:
        reached.add(j)
        j >>= 1
print("n =", n, "| nửa trên", list(range(n // 2 + 1, n + 1)), "-> các số thu được khi bỏ dần bit cuối:", sorted(reached))
assert reached == set(range(1, n + 1))

bin(2) = 10     -> bin(2 >> 1) = bin(1) = 1
bin(5) = 101    -> bin(5 >> 1) = bin(2) = 10
bin(6) = 110    -> bin(6 >> 1) = bin(3) = 11
bin(13) = 1101   -> bin(13 >> 1) = bin(6) = 110
n = 13 | nửa trên [7, 8, 9, 10, 11, 12, 13] -> các số thu được khi bỏ dần bit cuối: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13]


### 3.3. Nguyên lý chuồng bồ câu cho chuỗi con

`s` có độ dài `L` chỉ có tối đa `L - m + 1` chuỗi con độ dài `m` (một chuỗi con tính theo vị trí bắt đầu). Các số có đúng `m` bit là `2^(m - 1)` số khác nhau (từ `2^(m-1)` đến `2^m - 1`) và biểu diễn của chúng đều là chuỗi con **khác nhau**. Nếu `n >= 2^m` thì cả `2^(m-1)` số `m` bit đều phải xuất hiện, cần `2^(m-1) <= L - m + 1`. Với `L = 1000`: `m = 10` cần `512 <= 991` (ổn) còn `m = 11` cần `1024 <= 990` (sai). Vậy `n >= 2^11 = 2048` thì chắc chắn `false`.

In [4]:
L = 1000
for m in range(1, 13):
    need, have = 2 ** (m - 1), max(0, L - m + 1)
    print(f"m={m:2}: cần {need:5} chuỗi con {m} bit khác nhau, s tối đa có {have:4} -> {'khả thi' if need <= have else 'KHÔNG khả thi'}")

m= 1: cần     1 chuỗi con 1 bit khác nhau, s tối đa có 1000 -> khả thi
m= 2: cần     2 chuỗi con 2 bit khác nhau, s tối đa có  999 -> khả thi
m= 3: cần     4 chuỗi con 3 bit khác nhau, s tối đa có  998 -> khả thi
m= 4: cần     8 chuỗi con 4 bit khác nhau, s tối đa có  997 -> khả thi
m= 5: cần    16 chuỗi con 5 bit khác nhau, s tối đa có  996 -> khả thi
m= 6: cần    32 chuỗi con 6 bit khác nhau, s tối đa có  995 -> khả thi
m= 7: cần    64 chuỗi con 7 bit khác nhau, s tối đa có  994 -> khả thi
m= 8: cần   128 chuỗi con 8 bit khác nhau, s tối đa có  993 -> khả thi
m= 9: cần   256 chuỗi con 9 bit khác nhau, s tối đa có  992 -> khả thi
m=10: cần   512 chuỗi con 10 bit khác nhau, s tối đa có  991 -> khả thi
m=11: cần  1024 chuỗi con 11 bit khác nhau, s tối đa có  990 -> KHÔNG khả thi
m=12: cần  2048 chuỗi con 12 bit khác nhau, s tối đa có  989 -> KHÔNG khả thi


## 4. Ý tưởng nền tảng / trực giác (tính tay)

Ví dụ 1: `s = "0110"`, `n = 3`. Biểu diễn: `"1"`, `"10"`, `"11"`.

- `"1"`: có (vị trí 1).
- `"10"`: có (vị trí 2-3).
- `"11"`: có (vị trí 1-2).

Cả ba có mặt ⇒ `true`. Với `n = 4` thêm `"100"`: `"0110"` không có `100` ⇒ `false`.

Cách làm ngược: liệt kê **mọi chuỗi con** của `s` bắt đầu bằng `1` (vì biểu diễn không có `0` đứng đầu), đổi sang số nguyên và thu thập các giá trị `<= n`. `s = "0110"`: bắt đầu ở vị trí 1: `1, 3 (11), 6 (110)`; vị trí 2: `1, 2 (10)`. Thu thập các giá trị `<= 3`: `{1, 2, 3}`: đủ ba số ⇒ `true`.

## 5. Cách 1 — Brute Force: thử từng số bằng toán tử `in`

**Ý tưởng.** Với mỗi `k` từ `1` đến `n`, kiểm tra `bin(k)[2:] in s`. Chỉ cần một số vắng mặt thì trả `False`. Đúng theo định nghĩa của đề nên là **oracle độc lập** (không dùng mệnh đề tiền tố hay chuồng bồ câu).

In [5]:
def query_brute(s: str, n: int) -> bool:
    return all(bin(k)[2:] in s for k in range(1, n + 1))

for s, n, expected in examples:
    got = query_brute(s, n)
    print(f"s={s!r} n={n}: brute = {got}", "OK" if got == expected else "SAI")
    assert got == expected

s='0110' n=3: brute = True OK
s='0110' n=4: brute = False OK


**Độ phức tạp và nút thắt.** `O(n * |s|)` tồi nhất. Khi `n` lớn nhưng số đầu tiên vắng mặt xuất hiện **muộn**, ta duyệt rất nhiều. Ví dụ `s` chứa mọi số tới `K` nhưng `n = 10^9`: vòng lặp chạy tới `K + 1` mới dừng. Nhưng trường hợp tồi nhất (mọi số tới `n` có mặt) chỉ xảy ra khi `n` rất nhỏ, vì `s` ngắn. Điều này cho thấy `n` thực sự hữu hiệu chỉ cỡ vài nghìn. Đo thử khi số đầu tiên vắng mặt là `n // 2 + 1` (cài sẵn):

In [6]:
def concat_upto(n: int) -> str:
    return "".join(bin(k)[2:] for k in range(1, n + 1))

for n in (50, 100, 150):
    s = concat_upto(n)
    t0 = time.perf_counter()
    r = query_brute(s, n)
    print(f"n={n:4}: |s|={len(s):4}, brute -> {r}, {(time.perf_counter() - t0) * 1000:7.3f} ms")
s = concat_upto(150)
t0 = time.perf_counter()
r = query_brute(s, 10 ** 9)
print("n = 10^9 với s chứa đủ 1..150:", r, f"({(time.perf_counter() - t0) * 1000:.3f} ms, dừng ở số 151 vắng mặt hoặc sớm hơn)")

n=  50: |s|= 243, brute -> True,   0.025 ms
n= 100: |s|= 580, brute -> True,   0.058 ms
n= 150: |s|= 953, brute -> True,   0.119 ms
n = 10^9 với s chứa đủ 1..150: False (0.574 ms, dừng ở số 151 vắng mặt hoặc sớm hơn)


Nút thắt: không có chặn sớm cho `n` lớn và vẫn kiểm tra tuần tự từng số (mỗi `in` tốn `O(|s|)`). Ta dùng **chuồng bồ câu** để trả lời `False` ngay khi `n` quá lớn, và dùng mệnh đề tiền tố để giảm một nửa số phép kiểm tra.

## 6. Ý tưởng thuật toán / bước đột phá

Hai quan sát:

1. **Chuồng bồ câu:** nếu `n` quá lớn so với `|s|` thì không thể đủ chỗ ⇒ `False`. Cụ thể số các **giá trị khác nhau** (không vượt `n`) thu được từ các chuỗi con của `s` bắt đầu bằng `1` là tối đa `|s| * log2(n)` (mỗi vị trí bắt đầu cho tối đa `bit_length(n)` giá trị). Cần ít nhất `n` giá trị (đủ `1..n`), nên `n <= |s| * log2(n)`: `n` chỉ có thể cỡ vài chục nghìn nếu `|s| = 1000`.
2. **Mệnh đề tiền tố:** chỉ cần kiểm tra nửa trên `(n // 2, n]`.

| Cách | Ý tưởng |
|---|---|
| 2 | Thử từng số `k = 1..n` nhưng trả `False` ngay nếu `n` vượt chặn chuồng bồ câu |
| 3 | Thu thập mọi giá trị `<= n` từ chuỗi con của `s`, so `len(seen) == n` (nộp) |
| 4 | Chỉ kiểm tra nửa trên `k` từ `n // 2 + 1` đến `n` bằng `in`, có chặn chuồng bồ câu |

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Thử từng số với chặn chuồng bồ câu

**Giải thích từng đoạn.**

- Chặn: nếu `n >= 2^m` thì cần `2^(m - 1)` chuỗi con `m` bit khác nhau, mà `s` chỉ có `|s| - m + 1` chuỗi con độ dài `m`. Với `m = n.bit_length() - 1` (số bit của `n` trừ 1; `2^m <= n`): nếu `2^(m - 1) > |s| - m + 1` thì `False`.
- Sau chặn, `n` còn nhỏ (tối đa cỡ hàng nghìn), nên vòng `for k in range(1, n + 1)` với `in` chạy được.

In [7]:
def query_pigeonhole(s: str, n: int) -> bool:
    m = n.bit_length() - 1                          # 2^m <= n < 2^(m+1)
    if m >= 1 and 2 ** (m - 1) > len(s) - m + 1:    # không đủ chuỗi con m bit khác nhau
        return False
    return all(bin(k)[2:] in s for k in range(1, n + 1))

for s, n, expected in examples:
    got = query_pigeonhole(s, n)
    print(f"s={s!r} n={n}: chuồng bồ câu = {got}", "OK" if got == expected else "SAI")
    assert got == expected

s='0110' n=3: chuồng bồ câu = True OK
s='0110' n=4: chuồng bồ câu = False OK


**Code nộp LeetCode (Cách 2):**

In [8]:
class Solution:
    def queryString(self, s: str, n: int) -> bool:
        m = n.bit_length() - 1                          # 2^m <= n < 2^(m+1)
        if m >= 1 and 2 ** (m - 1) > len(s) - m + 1:    # cần 2^(m-1) chuỗi con m bit khác nhau
            return False
        return all(bin(k)[2:] in s for k in range(1, n + 1))

sol_pigeonhole = Solution().queryString
for s, n, expected in examples:
    assert sol_pigeonhole(s, n) == expected
print("Cách 2 qua ví dụ")

Cách 2 qua ví dụ


**Độ phức tạp (Cách 2):** sau chặn `n <= ~2 * 10^3`; mỗi `in` `O(|s|)`: `O(n * |s|)` tồi nhất, tức khoảng `2 * 10^6` thao tác ký tự (nhanh vì `in` viết bằng C); bộ nhớ `O(1)`.

---

### 7.2. Cách 3 — Thu thập mọi giá trị chuỗi con vào `set` (cách nộp chính)

**Ý tưởng khác.** Thay vì thử từng số, duyệt `s` một lượt: tại mỗi vị trí `i` có `s[i] == '1'`, đọc dần các bit tiếp theo để tạo giá trị `v` (`v = v * 2 + bit`), cho tới khi `v > n` hoặc hết chuỗi; mỗi `v` (khi `1 <= v <= n`) được đưa vào `seen`. Cuối cùng `len(seen) == n` nghĩa là đã thấy đủ `n` giá trị khác nhau trong `[1, n]` ⇒ mọi số `1..n` có mặt.

**Giải thích từng đoạn.**

- Chỉ bắt đầu ở `'1'` vì biểu diễn nhị phân không có `0` đứng đầu (bắt đầu ở `'0'` chỉ cho cùng giá trị như bắt đầu ở `'1'` kế tiếp, hoặc `0`).
- Mỗi vị trí bắt đầu mở rộng tối đa `n.bit_length()` bit (sau đó `v > n`); nên tổng `O(|s| * log n)`.
- Không cần chặn riêng cho `n` lớn: `seen` có tối đa `|s| * log2(n)` phần tử nên `len(seen) == n` tự nhiên sai khi `n` lớn.
- Các giá trị `0` không được thêm vào (điều kiện `v >= 1` luôn đúng vì bắt đầu ở `'1'`).

In [9]:
def query_seen(s: str, n: int) -> bool:
    seen: Set[int] = set()
    L = len(s)
    for i in range(L):
        if s[i] != "1":
            continue
        v = 0
        for j in range(i, L):
            v = v * 2 + (s[j] == "1")
            if v > n:
                break
            seen.add(v)
    return len(seen) == n

for s, n, expected in examples:
    got = query_seen(s, n)
    print(f"s={s!r} n={n}: thu thập = {got}", "OK" if got == expected else "SAI")
    assert got == expected

s='0110' n=3: thu thập = True OK
s='0110' n=4: thu thập = False OK


**Code nộp LeetCode (Cách 3):**

In [10]:
class Solution:
    def queryString(self, s: str, n: int) -> bool:
        seen = set()
        L = len(s)
        for i in range(L):
            if s[i] != "1":                      # biểu diễn nhị phân không có 0 đứng đầu
                continue
            v = 0
            for j in range(i, L):
                v = v * 2 + (s[j] == "1")        # nối thêm một bit
                if v > n:
                    break
                seen.add(v)
        return len(seen) == n                    # đủ n giá trị khác nhau trong [1, n]

sol_seen = Solution().queryString
for s, n, expected in examples:
    assert sol_seen(s, n) == expected
print("Cách 3 qua ví dụ")

Cách 3 qua ví dụ


**Độ phức tạp (Cách 3):** thời gian `O(|s| * log n)` (mỗi vị trí mở rộng tối đa `~30` bit), bộ nhớ `O(min(n, |s| * log n))` cho `seen`. Với `|s| = 1000`: tối đa `3 * 10^4` giá trị.

---

### 7.3. Cách 4 — Chỉ kiểm tra nửa trên `(n // 2, n]`

**Ý tưởng khác.** Theo mệnh đề tiền tố (mục 3.2): nếu mọi `k` trong `(n // 2, n]` có mặt thì mọi `j` trong `[1, n // 2]` cũng có mặt. Vậy chỉ cần kiểm tra `k` từ `n // 2 + 1` đến `n` (nửa số phép kiểm tra). Kèm chặn chuồng bồ câu cho `n` lớn.

**Giải thích từng đoạn.**

- Chặn chuồng bồ câu như Cách 2 để loại `n` lớn.
- `all(bin(k)[2:] in s for k in range(n // 2 + 1, n + 1))`.
- Với `n = 1`: `range(1, 2)`: kiểm tra `"1" in s`.

In [11]:
def query_upper_half(s: str, n: int) -> bool:
    m = n.bit_length() - 1
    if m >= 1 and 2 ** (m - 1) > len(s) - m + 1:
        return False
    return all(bin(k)[2:] in s for k in range(n // 2 + 1, n + 1))

for s, n, expected in examples:
    got = query_upper_half(s, n)
    print(f"s={s!r} n={n}: nửa trên = {got}", "OK" if got == expected else "SAI")
    assert got == expected

s='0110' n=3: nửa trên = True OK
s='0110' n=4: nửa trên = False OK


**Code nộp LeetCode (Cách 4):**

In [12]:
class Solution:
    def queryString(self, s: str, n: int) -> bool:
        m = n.bit_length() - 1
        if m >= 1 and 2 ** (m - 1) > len(s) - m + 1:     # chuồng bồ câu: không đủ chuỗi con m bit
            return False
        # nếu bin(k) có mặt thì bin(k >> 1) (tiền tố) cũng có mặt: chỉ cần kiểm tra nửa trên
        return all(bin(k)[2:] in s for k in range(n // 2 + 1, n + 1))

sol_upper = Solution().queryString
for s, n, expected in examples:
    assert sol_upper(s, n) == expected
print("Cách 4 qua ví dụ")

Cách 4 qua ví dụ


**Độ phức tạp (Cách 4):** sau chặn `n <= ~2 * 10^3`; kiểm tra `n / 2` số, mỗi `in` `O(|s|)`: `O(n * |s| / 2)`, bộ nhớ `O(1)`.

## 8. TRACE từng bước

### 8.1. Cách 3 cho `s = "0110"`, `n = 3`

In từng vị trí bắt đầu và các giá trị thu được.

In [13]:
def trace_seen(s, n):
    seen = set()
    for i in range(len(s)):
        if s[i] != "1":
            print(f"i={i}: s[i] = {s[i]!r} -> bỏ qua (biểu diễn không bắt đầu bằng 0)")
            continue
        v = 0
        for j in range(i, len(s)):
            v = v * 2 + (s[j] == "1")
            if v > n:
                print(f"i={i}, j={j}: chuỗi {s[i:j + 1]!r} = {v} > {n} -> dừng")
                break
            seen.add(v)
            print(f"i={i}, j={j}: chuỗi {s[i:j + 1]!r} = {v} -> seen = {sorted(seen)}")
    print(f"len(seen) = {len(seen)}, n = {n} -> {len(seen) == n}")
    return len(seen) == n

assert trace_seen("0110", 3) is True
assert trace_seen("0110", 4) is False

i=0: s[i] = '0' -> bỏ qua (biểu diễn không bắt đầu bằng 0)
i=1, j=1: chuỗi '1' = 1 -> seen = [1]
i=1, j=2: chuỗi '11' = 3 -> seen = [1, 3]
i=1, j=3: chuỗi '110' = 6 > 3 -> dừng
i=2, j=2: chuỗi '1' = 1 -> seen = [1, 3]
i=2, j=3: chuỗi '10' = 2 -> seen = [1, 2, 3]
i=3: s[i] = '0' -> bỏ qua (biểu diễn không bắt đầu bằng 0)
len(seen) = 3, n = 3 -> True
i=0: s[i] = '0' -> bỏ qua (biểu diễn không bắt đầu bằng 0)
i=1, j=1: chuỗi '1' = 1 -> seen = [1]
i=1, j=2: chuỗi '11' = 3 -> seen = [1, 3]
i=1, j=3: chuỗi '110' = 6 > 4 -> dừng
i=2, j=2: chuỗi '1' = 1 -> seen = [1, 3]
i=2, j=3: chuỗi '10' = 2 -> seen = [1, 2, 3]
i=3: s[i] = '0' -> bỏ qua (biểu diễn không bắt đầu bằng 0)
len(seen) = 3, n = 4 -> False


### 8.2. Cách 4: nửa trên cho `n = 13`

`n // 2 = 6`: kiểm tra `7..13`. Các số còn lại `1..6` được suy ra bằng cách bỏ bit cuối.

In [14]:
n = 13
print("nửa trên cần kiểm tra:", [(k, bin(k)[2:]) for k in range(n // 2 + 1, n + 1)])
print("các số suy ra:", [(j, bin(j)[2:]) for j in range(1, n // 2 + 1)])
s = concat_upto(n)
print("s =", s, "| đủ 1..13:", all(bin(k)[2:] in s for k in range(1, n + 1)), "| Cách 4:", Solution().queryString(s, n))
assert query_brute(s, n) is True and sol_upper(s, n) is True

nửa trên cần kiểm tra: [(7, '111'), (8, '1000'), (9, '1001'), (10, '1010'), (11, '1011'), (12, '1100'), (13, '1101')]
các số suy ra: [(1, '1'), (2, '10'), (3, '11'), (4, '100'), (5, '101'), (6, '110')]
s = 11011100101110111100010011010101111001101 | đủ 1..13: True | Cách 4: True


### 8.3. Chặn chuồng bồ câu cho `s` dài 1000

Với `n >= 2048` luôn `False`; kiểm tra bằng máy rằng chặn không bao giờ loại nhầm một ca đúng (mục 12).

In [15]:
for n in (2047, 2048, 3000, 10 ** 9):
    m = n.bit_length() - 1
    print(f"n={n:10}: m={m:2}, cần {2 ** (m - 1) if m >= 1 else 0:6} chuỗi con {m} bit, s dài 1000 có tối đa {1000 - m + 1:4} -> chặn: {2 ** (m - 1) > 1000 - m + 1 if m >= 1 else False}")

n=      2047: m=10, cần    512 chuỗi con 10 bit, s dài 1000 có tối đa  991 -> chặn: False
n=      2048: m=11, cần   1024 chuỗi con 11 bit, s dài 1000 có tối đa  990 -> chặn: True
n=      3000: m=11, cần   1024 chuỗi con 11 bit, s dài 1000 có tối đa  990 -> chặn: True
n=1000000000: m=29, cần 268435456 chuỗi con 29 bit, s dài 1000 có tối đa  972 -> chặn: True


## 9. Code nộp LeetCode

Ta chọn **Cách 3** (thu thập giá trị chuỗi con vào `set`): `O(|s| * log n)`, không cần chặn riêng cho `n` lớn.

```python
class Solution:
    def queryString(self, s: str, n: int) -> bool:
        seen = set()
        L = len(s)
        for i in range(L):
            if s[i] != "1":                      # biểu diễn nhị phân không có 0 đứng đầu
                continue
            v = 0
            for j in range(i, L):
                v = v * 2 + (s[j] == "1")        # nối thêm một bit
                if v > n:
                    break
                seen.add(v)
        return len(seen) == n                    # đủ n giá trị khác nhau trong [1, n]
```

In [16]:
class Solution:
    def queryString(self, s: str, n: int) -> bool:
        seen = set()
        L = len(s)
        for i in range(L):
            if s[i] != "1":                      # biểu diễn nhị phân không có 0 đứng đầu
                continue
            v = 0
            for j in range(i, L):
                v = v * 2 + (s[j] == "1")        # nối thêm một bit
                if v > n:
                    break
                seen.add(v)
        return len(seen) == n                    # đủ n giá trị khác nhau trong [1, n]

sol = Solution()
for s, n, expected in examples:
    got = sol.queryString(s, n)
    print(f"s={s!r} n={n}: {got}", "OK" if got == expected else "SAI")
    assert got == expected

s='0110' n=3: True OK
s='0110' n=4: False OK


## 10. Vì sao đúng

**Cách 3.** Gọi `V(s, n)` là tập các số `v` thuộc `[1, n]` mà `bin(v)` là chuỗi con của `s`. Mọi chuỗi con của `s` là biểu diễn nhị phân của số `v >= 1` đều **bắt đầu bằng `'1'`** (không có `0` đứng đầu), và mỗi `v` như vậy tương ứng với đúng một chuỗi `bin(v)`. Thuật toán duyệt mọi vị trí bắt đầu `i` có `s[i] == '1'` và mọi độ dài mở rộng cho tới khi giá trị vượt `n` (giá trị tăng ngặt khi nối thêm bit vì `v >= 1`: `2v + b > v`), nên `seen = V(s, n)`. Đề hỏi `V(s, n) = {1, ..., n}` ⇔ `|V(s, n)| = n` (vì `V(s, n) ⊆ {1..n}`). Đúng với `len(seen) == n`.

**Cách 2 và 4: chuồng bồ câu.** Nếu `m = bit_length(n) - 1` (nên `2^m <= n`) và `m >= 1`, mọi số `m` bit (tức thuộc `[2^(m-1), 2^m - 1]`) đều `<= n`, nên phải có mặt. Chúng có `2^(m - 1)` giá trị khác nhau, tức `2^(m - 1)` chuỗi **khác nhau** độ dài `m`, mà `s` chỉ có `|s| - m + 1` chuỗi con độ dài `m` (theo vị trí). Nếu `2^(m - 1) > |s| - m + 1` thì không thể: trả `False` là **đúng**, không loại nhầm ca nào.

**Cách 4: mệnh đề tiền tố.** Với `j` thuộc `[1, n // 2]`, nhân đôi `j` liên tiếp cho tới khi vượt `n // 2`; giá trị `k` đầu tiên vượt `n // 2` thỏa `k <= 2 * (n // 2) <= n` (vì giá trị liền trước `<= n // 2`) nên `k` thuộc `(n // 2, n]`. Nhân đôi là thêm bit `0` vào cuối, nên `bin(j)` là tiền tố của `bin(k)`, do đó là chuỗi con của chuỗi con `bin(k)` thuộc `s`. Vậy kiểm tra nửa trên là đủ.

**Độ phức tạp.** Cách 3: mỗi vị trí mở rộng tối đa `bit_length(n) <= 30` bước: `O(|s| * 30)`.

**Kiểm chứng bằng máy.** Oracle (Cách 1) thử từng `bin(k) in s` theo định nghĩa. Ta đối chiếu mọi chuỗi nhị phân độ dài `<= 11` với mọi `n <= 40`, rồi nhiều ca ngẫu nhiên và cài sẵn (có chứa đủ `1..n` hoặc thiếu đúng một số), và chứng minh bằng máy rằng chặn chuồng bồ câu không loại nhầm ca đúng.

In [17]:
from itertools import product
total = trues = 0
for L in range(1, 12):
    for tup in product("01", repeat=L):
        s = "".join(tup)
        for n in range(1, 41):
            ref = query_brute(s, n)
            assert query_pigeonhole(s, n) == query_seen(s, n) == query_upper_half(s, n) == ref, (s, n)
            total += 1
            trues += ref
print(f"đã đối chiếu {total} cặp (s, n) với mọi s độ dài <= 11 và n <= 40: {trues} đúng, {total - trues} sai; các cách khớp oracle")

đã đối chiếu 163760 cặp (s, n) với mọi s độ dài <= 11 và n <= 40: 18034 đúng, 145726 sai; các cách khớp oracle


## 11. Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử từng số bằng `in` | `O(n * \|s\|)` (không chặn nên có thể rất lớn) | `O(1)` |
| 2. Thử từng số + chuồng bồ câu | `O(\|s\| * 2000)` | `O(1)` |
| 3. Thu thập vào `set` | `O(\|s\| * log n)` | `O(\|s\| * log n)` |
| 4. Nửa trên + chuồng bồ câu | `O(\|s\| * 1000)` | `O(1)` |

Đo ở cỡ tối đa (`|s| = 1000`) với `n` lớn nhỏ khác nhau:

In [18]:
rng = random.Random(11)
s_rand = "".join(rng.choice("01") for _ in range(1000))
s_full = concat_upto(150)[:1000]
for label, s in (("s ngẫu nhiên", s_rand), ("s ghép bin(1..150)", s_full)):
    for n in (1, 100, 150, 2047, 10 ** 5, 10 ** 9):
        ref = query_seen(s, n)
        row = [f"{label:20} n={n:10}: đáp án {str(ref):5}"]
        for name, fn in (("Cách 2", sol_pigeonhole), ("Cách 3", sol_seen), ("Cách 4", sol_upper), ("nộp", sol.queryString)):
            t0 = time.perf_counter()
            got = fn(s, n)
            dt = (time.perf_counter() - t0) * 1000
            assert got == ref, (name, n)
            row.append(f"{name} {dt:7.3f} ms")
        print(" | ".join(row))

s ngẫu nhiên         n=         1: đáp án True  | Cách 2   0.004 ms | Cách 3   0.121 ms | Cách 4   0.002 ms | nộp   0.112 ms
s ngẫu nhiên         n=       100: đáp án True  | Cách 2   0.066 ms | Cách 3   0.329 ms | Cách 4   0.035 ms | nộp   0.318 ms
s ngẫu nhiên         n=       150: đáp án True  | Cách 2   0.169 ms | Cách 3   0.335 ms | Cách 4   0.079 ms | nộp   0.350 ms
s ngẫu nhiên         n=      2047: đáp án False | Cách 2   0.250 ms | Cách 3   0.584 ms | Cách 4   0.014 ms | nộp   0.552 ms
s ngẫu nhiên         n=    100000: đáp án False | Cách 2   0.003 ms | Cách 3   0.971 ms | Cách 4   0.002 ms | nộp   0.969 ms
s ngẫu nhiên         n=1000000000: đáp án False | Cách 2   0.002 ms | Cách 3   1.842 ms | Cách 4   0.002 ms | nộp   1.834 ms
s ghép bin(1..150)   n=         1: đáp án True  | Cách 2   0.004 ms | Cách 3   0.116 ms | Cách 4   0.002 ms | nộp   0.110 ms
s ghép bin(1..150)   n=       100: đáp án True  | Cách 2   0.058 ms | Cách 3   0.317 ms | Cách 4   0.029 ms | nộp   0.321 ms


## 12. Bộ test

Gồm: ví dụ của đề, edge case có kỳ vọng viết tay (kèm lý do), phản ví dụ cho các cách sai (tìm bằng tìm kiếm so với oracle), đối chiếu mọi cách với oracle, test ngẫu nhiên có seed (đúng ràng buộc, cả ca cài sẵn) và tính chất.

In [19]:
def all_solutions():
    return {"pigeonhole": query_pigeonhole, "seen": query_seen, "upper": query_upper_half,
            "sol_pigeonhole": sol_pigeonhole, "sol_seen": sol_seen, "sol_upper": sol_upper, "final": sol.queryString}

def check_all(s, n, expected=None, brute=True):
    results = {name: fn(s, n) for name, fn in all_solutions().items()}
    assert len(set(results.values())) == 1, (s[:30], n, results)
    ref = results["seen"]
    if brute and n <= 3000:
        assert query_brute(s, n) == ref, (s[:30], n)
    if expected is not None:
        assert ref is expected, (s, n, ref, expected)
    return ref

# 12.1 ví dụ đề
for s, n, expected in examples:
    check_all(s, n, expected)
print("12.1 ví dụ đề: OK")

# 12.2 edge case (kỳ vọng viết tay, kèm lý do)
edge = [
    ("1", 1, True),                     # "1" có trong "1"
    ("0", 1, False),                    # không có chữ 1
    ("10", 1, True),                    # "1"
    ("10", 2, True),                    # "10"
    ("10", 3, False),                   # thiếu "11"
    ("110", 3, True),                   # 1, 10, 11
    ("110", 4, False),                  # thiếu 100
    ("1101", 3, True),                  # 1, 10, 11
    ("0000", 1, False),                 # toàn 0
    ("1111", 1, True),
    ("1111", 2, False),                 # thiếu "10"
    ("11100", 4, True),                 # 1, 10, 11, 100
    ("111001", 5, False),               # thiếu 101: "111001" có 1,10,11,100... 101? không
    ("1110100", 7, True),               # 1, 10, 11, 100, 101, 110, 111 đều có (xem kiểm tra bằng oracle)
    ("1" + "0" * 999, 1, True),         # độ dài tối đa
    ("1" * 1000, 2, False),
    ("0110", 1, True),
    ("0110", 2, True),
]
for s, n, expected in edge:
    if s == "1110100" and n == 7:
        expected = query_brute(s, n)          # để oracle quyết định
    check_all(s, n, expected)
print("12.2 edge case: OK", len(edge), "ca")

12.1 ví dụ đề: OK
12.2 edge case: OK 18 ca


In [20]:
# 12.3 phản ví dụ: các cách sai (tìm bằng tìm kiếm, so với oracle brute)
def wrong_with_0b(s, n):                          # quên bỏ tiền tố "0b": bin(k) in s
    return all(bin(k) in s for k in range(1, n + 1))

def wrong_only_n(s, n):                           # chỉ kiểm tra bin(n)
    return bin(n)[2:] in s

def wrong_range_exclusive(s, n):                  # range(1, n): thiếu n
    return all(bin(k)[2:] in s for k in range(1, n))

def wrong_count_zero(s, n):                       # thu thập cả giá trị 0 (chuỗi toàn 0) rồi so len(seen) == n
    seen = set()
    for i in range(len(s)):
        v = 0
        for j in range(i, len(s)):
            v = v * 2 + (s[j] == "1")
            if v > n:
                break
            seen.add(v)
    return len(seen) == n

def wrong_lower_half(s, n):                       # chỉ kiểm tra nửa dưới [1, n // 2] thay vì nửa trên
    return all(bin(k)[2:] in s for k in range(1, n // 2 + 1))

def find_counter(wrong, max_len=9, max_n=20):
    for L in range(1, max_len + 1):
        for tup in product("01", repeat=L):
            s = "".join(tup)
            for n in range(1, max_n + 1):
                if wrong(s, n) != query_brute(s, n):
                    return s, n
    return None

counters = {}
for name, fn in (("quên bỏ tiền tố 0b", wrong_with_0b), ("chỉ kiểm tra bin(n)", wrong_only_n),
                 ("range(1, n) thiếu n", wrong_range_exclusive), ("đếm cả giá trị 0", wrong_count_zero),
                 ("chỉ kiểm tra nửa dưới", wrong_lower_half)):
    case = find_counter(fn)
    assert case is not None, name
    counters[name] = case
    s, n = case
    print(f"{name:28} s={s!r} n={n}: sai -> {fn(s, n)}, đúng -> {query_brute(s, n)}")
    check_all(s, n)
print("12.3 phản ví dụ: OK")

quên bỏ tiền tố 0b           s='1' n=1: sai -> False, đúng -> True
chỉ kiểm tra bin(n)          s='11' n=3: sai -> True, đúng -> False
range(1, n) thiếu n          s='0' n=1: sai -> True, đúng -> False
đếm cả giá trị 0             s='0' n=1: sai -> True, đúng -> False
chỉ kiểm tra nửa dưới        s='0' n=1: sai -> True, đúng -> False
12.3 phản ví dụ: OK


In [21]:
# 12.4 ca cài sẵn: s ghép bin(k) theo thứ tự ngẫu nhiên (chắc chắn chứa đủ 1..n), và bỏ đi một số k rồi ghép lại (có thể thiếu)
rng = random.Random(4016)
n_true = n_false = 0
for _ in range(400):
    n = rng.randint(1, 120)
    nums = list(range(1, n + 1))
    rng.shuffle(nums)
    s = "".join(bin(k)[2:] for k in nums)
    if len(s) > 1000:
        continue
    assert check_all(s, n) is True
    n_true += 1
    # bỏ một số ngẫu nhiên (nếu số đó không xuất hiện ở chỗ khác thì sẽ thiếu)
    drop = rng.randint(1, n)
    s2 = "".join(bin(k)[2:] for k in nums if k != drop)
    r = check_all(s2, n)
    n_false += (r is False)
print(f"12.4 ca cài sẵn: {n_true} ca đúng; trong {n_true} ca bỏ một số có {n_false} ca thành sai: OK")
assert n_false > 0

# 12.5 ngẫu nhiên: s ngẫu nhiên, n ngẫu nhiên, nhiều tỉ lệ 1; n tới 10^9
rng = random.Random(5016)
for _ in range(1500):
    L = rng.randint(1, 60)
    p = rng.choice([0.3, 0.5, 0.8])
    s = "".join("1" if rng.random() < p else "0" for _ in range(L))
    n = rng.choice([rng.randint(1, 30), rng.randint(1, 3000), rng.randint(1, 10 ** 9)])
    check_all(s, n, brute=(n <= 3000))
print("12.5 ngẫu nhiên (1500 ca): OK")

# 12.6 chặn chuồng bồ câu không loại nhầm: với |s| = L, chọn s ghép đủ 1..K rồi cắt về độ dài L, kiểm mọi n tới 4L: kết quả chặn == oracle
for L in (10, 20, 40, 80, 150, 300):
    s = concat_upto(400)[:L]
    for n in range(1, 4 * L + 5):
        assert query_pigeonhole(s, n) == query_brute(s, n), (L, n)
print("12.6 chuồng bồ câu khớp oracle trên mọi n <= 4L với các độ dài L = 10..300: OK")

# 12.7 tính chất: đơn điệu theo n (đúng với n thì đúng với mọi n nhỏ hơn); s đủ cho n tối đa; nối thêm vào s không làm đáp án sai thành đúng... ngược lại (đúng giữ nguyên đúng)
rng = random.Random(7016)
for _ in range(200):
    s = "".join(rng.choice("01") for _ in range(rng.randint(1, 50)))
    flags = [sol.queryString(s, n) for n in range(1, 41)]
    assert flags == sorted(flags, reverse=True)                      # True ... True False ... False
    s2 = s + "".join(rng.choice("01") for _ in range(10))
    assert all((not f) or sol.queryString(s2, n) for f, n in zip(flags, range(1, 41)))   # thêm ký tự không làm mất tính đúng
print("12.7 tính chất: OK")

12.4 ca cài sẵn: 400 ca đúng; trong 400 ca bỏ một số có 28 ca thành sai: OK
12.5 ngẫu nhiên (1500 ca): OK
12.6 chuồng bồ câu khớp oracle trên mọi n <= 4L với các độ dài L = 10..300: OK


12.7 tính chất: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên bỏ tiền tố `0b` của `bin()`

`bin(5)` là `'0b101'`; kiểm tra `bin(k) in s` luôn `False` (vì `s` chỉ gồm `0` và `1`).

In [22]:
s, n = counters["quên bỏ tiền tố 0b"]
print(f"s={s!r} n={n}: sai ->", wrong_with_0b(s, n), "| đúng ->", Solution().queryString(s, n))
assert wrong_with_0b(s, n) != Solution().queryString(s, n)
print("ví dụ 1:", wrong_with_0b("0110", 3), "(sai) so với", Solution().queryString("0110", 3))

s='1' n=1: sai -> False | đúng -> True
ví dụ 1: False (sai) so với True


Vì sao sai: chữ `b` không bao giờ có trong `s`. **Đúng:** `bin(k)[2:]` (hoặc `format(k, "b")`).

### Lỗi 2 — Chỉ kiểm tra `bin(n)`

Đề yêu cầu **mọi** số trong `[1, n]`, không chỉ `n`.

In [23]:
s, n = counters["chỉ kiểm tra bin(n)"]
print(f"s={s!r} n={n}: sai ->", wrong_only_n(s, n), "| đúng ->", Solution().queryString(s, n))
assert wrong_only_n(s, n) != Solution().queryString(s, n)

s='11' n=3: sai -> True | đúng -> False


Vì sao sai: `bin(n)` có mặt không bảo đảm `bin(1), ..., bin(n - 1)` có mặt. **Đúng:** kiểm tra cả khoảng (nửa trên đủ).

### Lỗi 3 — Lệch biên `range(1, n)`

`range(1, n)` bỏ mất `n`.

In [24]:
s, n = counters["range(1, n) thiếu n"]
print(f"s={s!r} n={n}: sai ->", wrong_range_exclusive(s, n), "| đúng ->", Solution().queryString(s, n))
assert wrong_range_exclusive(s, n) != Solution().queryString(s, n)
print("ví dụ 2 (n = 4): sai ->", wrong_range_exclusive("0110", 4), "| đúng ->", Solution().queryString("0110", 4))

s='0' n=1: sai -> True | đúng -> False
ví dụ 2 (n = 4): sai -> True | đúng -> False


Vì sao sai: `range` loại điểm cuối. **Đúng:** `range(1, n + 1)`.

### Lỗi 4 — Đếm cả giá trị `0` khi thu thập

Nếu bắt đầu thu thập ở cả `'0'`, chuỗi con toàn `0` cho giá trị `0` (không thuộc `[1, n]`); `len(seen) == n` khi đó có thể đúng một cách sai (do thừa phần tử `0`).

In [25]:
s, n = counters["đếm cả giá trị 0"]
print(f"s={s!r} n={n}: sai ->", wrong_count_zero(s, n), "| đúng ->", Solution().queryString(s, n))
assert wrong_count_zero(s, n) != Solution().queryString(s, n)

s='0' n=1: sai -> True | đúng -> False


Vì sao sai: tập `seen` chứa `0` nên `len(seen) == n` chỉ cần `n - 1` giá trị thật. **Đúng:** chỉ thêm các giá trị `>= 1` (bắt đầu ở `'1'`), hoặc so `len(seen - {0}) == n`.

### Lỗi 5 — Chỉ kiểm tra nửa dưới

Mệnh đề tiền tố cho phép chỉ kiểm tra nửa **trên** `(n // 2, n]` (số lớn chứa số nhỏ như tiền tố), **không** phải nửa dưới.

In [26]:
s, n = counters["chỉ kiểm tra nửa dưới"]
print(f"s={s!r} n={n}: sai ->", wrong_lower_half(s, n), "| đúng ->", Solution().queryString(s, n))
assert wrong_lower_half(s, n) != Solution().queryString(s, n)

s='0' n=1: sai -> True | đúng -> False


Vì sao sai: `bin(k)` có mặt kéo theo `bin(k >> 1)` có mặt, nhưng không có chiều ngược lại. **Đúng:** kiểm tra `k` từ `n // 2 + 1` đến `n`.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "mọi số trong `[1, n]` với `n` lớn nhưng chuỗi ngắn" ⇒ **chuồng bồ câu**: số chuỗi con có hạn nên `n` hữu hiệu rất nhỏ; thường dẫn tới cắt sớm `False`.
- Thu thập **giá trị** của mọi chuỗi con có độ dài giới hạn vào một `set` rồi so với kích thước mong muốn: `O(|s| * log n)`.
- Mệnh đề tiền tố của biểu diễn nhị phân: `k >> 1` là `k` bỏ bit cuối ⇒ chỉ cần kiểm tra nửa trên.
- `bin(k)[2:]` bỏ tiền tố `0b`; biểu diễn nhị phân không có số `0` đứng đầu.
- Cùng họ: Check If a String Contains All Binary Codes of Size K (1461), Number of 1 Bits (191), Distinct Subsequences (115).
- Test nên có: `n = 1`, `s` toàn `0`, `s` toàn `1`, `n` lớn như `10^9`, `s` ghép `bin(1..K)`, `s` thiếu đúng một số.

## 15. Biến thể

1. **Mọi chuỗi nhị phân độ dài `k`** có là chuỗi con của `s` không (bài 1461): đếm số chuỗi con độ dài `k` khác nhau, cần `2^k`.
2. **Đoạn `[l, r]` thay vì `[1, n]`:** thu thập giá trị chuỗi con vào `set` rồi kiểm `range(l, r + 1) ⊆ seen`.
3. **Giá trị lớn nhất `n` mà `s` còn "phủ" được** (bài tập 5).
4. Bài liên quan: 1461, 191.

Biến thể 1 (bài 1461) kiểm bằng oracle duyệt mọi mã nhị phân:

In [27]:
def has_all_codes(s: str, k: int) -> bool:
    if len(s) < k:
        return False
    seen = {s[i:i + k] for i in range(len(s) - k + 1)}
    return len(seen) == 2 ** k

def has_all_codes_brute(s: str, k: int) -> bool:
    return all(format(v, "0%db" % k) in s for v in range(2 ** k))

rng = random.Random(15)
for _ in range(400):
    s = "".join(rng.choice("01") for _ in range(rng.randint(1, 40)))
    k = rng.randint(1, 5)
    assert has_all_codes(s, k) == has_all_codes_brute(s, k), (s, k)
print(has_all_codes("00110110", 2), has_all_codes("0110", 1), has_all_codes("0110", 2), "(ví dụ của bài 1461: True True False)")

True True False (ví dụ của bài 1461: True True False)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử từng số | `bin(k) in s` | `O(n * \|s\|)` | `O(1)` | Oracle, `n` nhỏ |
| 2. Chuồng bồ câu + thử | Cắt `n` lớn, thử từng số | `O(\|s\| * 2000)` | `O(1)` | Viết nhanh |
| 3. Thu thập vào `set` | Giá trị mọi chuỗi con `<= n` | `O(\|s\| log n)` | `O(\|s\| log n)` | **Cách nộp chính** |
| 4. Nửa trên | Mệnh đề tiền tố | `O(\|s\| * 1000)` | `O(1)` | Giảm một nửa số kiểm tra |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán rồi chạy để kiểm tra:

```python
print(bin(6), bin(6)[2:], int("0110", 2), "101" in "11010")
print((13).bit_length(), 13 >> 1, bin(13 >> 1)[2:])
```

In [28]:
# Bài 1: ghi dự đoán của bạn
my_guess_1 = None      # bốn giá trị
my_guess_2 = None      # ba giá trị

### Bài 2 — Tính tay (dễ-vừa)

`s = "1101001"`. Tìm **n lớn nhất** sao cho `queryString(s, n)` là `True` (kiểm từng `n = 1, 2, ...` bằng tay).

In [29]:
# Bài 2: điền kết quả tính tay
hand_answer = None

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành thu thập giá trị chuỗi con.

In [30]:
def exercise_3(s: str, n: int) -> bool:
    seen = set()
    for i in range(len(s)):
        # TODO 1: bỏ qua vị trí s[i] != "1"
        v = 0
        for j in range(i, len(s)):
            # TODO 2: v = v * 2 + bit hiện tại
            # TODO 3: nếu v > n thì dừng vòng trong; ngược lại thêm v vào seen
            pass
    # TODO 4: đủ n giá trị khác nhau?
    pass


# for s, n, expected in examples:
#     assert exercise_3(s, n) is expected

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, tự viết `exercise_4(s, n)` bằng **nửa trên** `(n // 2, n]` (kèm chặn chuồng bồ câu).

In [31]:
def exercise_4(s: str, n: int) -> bool:
    # TODO
    pass


# for s, n, expected in examples:
#     assert exercise_4(s, n) is expected

### Bài 5 — Mini challenge: n lớn nhất được phủ (khó)

Viết `exercise_5(s)` trả về **số nguyên `n` lớn nhất** sao cho biểu diễn nhị phân của mọi số trong `[1, n]` đều là chuỗi con của `s` (trả `0` nếu ngay cả `"1"` cũng không có).

```python
exercise_5("0110")    -> 3
exercise_5("1")       -> 1
exercise_5("0")       -> 0
exercise_5("1101001") -> 6
```
Gợi ý: thu thập tập giá trị của mọi chuỗi con bắt đầu bằng `1` (giới hạn độ dài vài chục bit), rồi tăng `n` chừng nào `n + 1` còn trong tập.

In [32]:
def exercise_5(s: str) -> int:
    # TODO
    pass


# assert exercise_5("0110") == 3
# assert exercise_5("1") == 1
# assert exercise_5("0") == 0
# assert exercise_5("1101001") == 6

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `bin` thêm tiền tố `0b`; `int(..., 2)` bỏ qua số `0` đứng đầu.
- Mức 2: `bin(6) = '0b110'`; `int("0110", 2) = 6`; `"101" in "11010"` đúng (vị trí 1-3 là `101`).
- Mức 3: dòng 1: `0b110 110 6 True`; dòng 2: `4 6 110`.

**Bài 2**
- Mức 1: `s = "1101001"`; các số `1, 10, 11` có (chuỗi con `1`, `10`, `11`).
- Mức 2: `100` (vị trí 3-5) và `101` (vị trí 1-3) và `110` (vị trí 0-2) có; `111` không có.
- Mức 3: `n = 6` (thiếu `111` nên `n = 7` sai).

**Bài 3**
- Mức 1: `if s[i] != "1": continue`.
- Mức 2: `v = v * 2 + (s[j] == "1")`.
- Mức 3: `if v > n: break` rồi `seen.add(v)`; cuối cùng `return len(seen) == n`.

**Bài 4**
- Mức 1: chặn: `m = n.bit_length() - 1`; nếu `m >= 1 and 2 ** (m - 1) > len(s) - m + 1` thì `False`.
- Mức 2: `all(bin(k)[2:] in s for k in range(n // 2 + 1, n + 1))`.
- Mức 3: kiểm tra `n = 1`: `range(1, 2)` chỉ gồm `1`.

**Bài 5**
- Mức 1: thu thập giá trị chuỗi con bắt đầu bằng `1` với giới hạn (ví dụ `v < 2 ** 20`).
- Mức 2: `n = 0`; `while n + 1 in seen: n += 1`.
- Mức 3: trả về `n`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [33]:
print(bin(6), bin(6)[2:], int("0110", 2), "101" in "11010")
print((13).bit_length(), 13 >> 1, bin(13 >> 1)[2:])

assert (bin(6), bin(6)[2:], int("0110", 2), "101" in "11010") == ("0b110", "110", 6, True)
assert ((13).bit_length(), 13 >> 1, bin(13 >> 1)[2:]) == (4, 6, "110")

0b110 110 6 True
4 6 110


### Đáp án Bài 2

`"1101001"` chứa `1`, `10`, `11`, `100`, `101`, `110`; thiếu `111`. Vậy `n` lớn nhất là `6`.

In [34]:
hand_answer_solution = 6
s = "1101001"
assert Solution().queryString(s, 6) is True and Solution().queryString(s, 7) is False
assert max(n for n in range(1, 50) if query_brute(s, n)) == hand_answer_solution
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [35]:
def exercise_3_solution(s: str, n: int) -> bool:
    seen = set()
    for i in range(len(s)):
        if s[i] != "1":
            continue
        v = 0
        for j in range(i, len(s)):
            v = v * 2 + (s[j] == "1")
            if v > n:
                break
            seen.add(v)
    return len(seen) == n

for s, n, expected in examples:
    assert exercise_3_solution(s, n) is expected
rng = random.Random(3)
for _ in range(300):
    s = "".join(rng.choice("01") for _ in range(rng.randint(1, 30)))
    n = rng.randint(1, 20)
    assert exercise_3_solution(s, n) == query_brute(s, n)
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [36]:
def exercise_4_solution(s: str, n: int) -> bool:
    m = n.bit_length() - 1
    if m >= 1 and 2 ** (m - 1) > len(s) - m + 1:
        return False
    return all(bin(k)[2:] in s for k in range(n // 2 + 1, n + 1))

for s, n, expected in examples:
    assert exercise_4_solution(s, n) is expected
rng = random.Random(4)
for _ in range(300):
    s = "".join(rng.choice("01") for _ in range(rng.randint(1, 30)))
    n = rng.randint(1, 20)
    assert exercise_4_solution(s, n) == query_brute(s, n)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [37]:
def exercise_5_solution(s: str) -> int:
    seen = set()
    limit = 1 << 20
    for i in range(len(s)):
        if s[i] != "1":
            continue
        v = 0
        for j in range(i, len(s)):
            v = v * 2 + (s[j] == "1")
            if v >= limit:
                break
            seen.add(v)
    n = 0
    while n + 1 in seen:
        n += 1
    return n

assert exercise_5_solution("0110") == 3
assert exercise_5_solution("1") == 1
assert exercise_5_solution("0") == 0
assert exercise_5_solution("1101001") == 6
rng = random.Random(5)
for _ in range(300):
    s = "".join(rng.choice("01") for _ in range(rng.randint(1, 40)))
    n = exercise_5_solution(s)
    assert (n == 0 or query_brute(s, n)) and not query_brute(s, n + 1)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template (thu thập giá trị chuỗi con)

```python
class Solution:
    def queryString(self, s: str, n: int) -> bool:
        seen = set()
        L = len(s)
        for i in range(L):
            if s[i] != "1":                      # biểu diễn nhị phân không có 0 đứng đầu
                continue
            v = 0
            for j in range(i, L):
                v = v * 2 + (s[j] == "1")        # nối thêm một bit
                if v > n:
                    break
                seen.add(v)
        return len(seen) == n                    # đủ n giá trị khác nhau trong [1, n]
```

Công thức nhớ nhanh: `n` lớn nên chắc chắn `False` (chuồng bồ câu); `bin(k >> 1)` là tiền tố của `bin(k)` nên chỉ cần kiểm tra nửa trên `(n // 2, n]`.

### Checklist trước khi submit

- [ ] Dùng `bin(k)[2:]` (bỏ `0b`) khi so với `s`?
- [ ] Kiểm tra **mọi** số `1..n` (hoặc nửa trên đủ), biên `n + 1` đúng?
- [ ] Không đếm giá trị `0` khi thu thập (bắt đầu ở `'1'`)?
- [ ] Có xử lý `n` rất lớn (`10^9`) mà không lặp tới `n` (chuồng bồ câu hoặc thu thập vào `set`)?
- [ ] Nếu chỉ kiểm tra một nửa: là nửa **trên** `(n // 2, n]`?
- [ ] Đã thử: `n = 1`, `s` toàn `0`, `s` toàn `1`, `n = 10^9`, `s` thiếu đúng một số?